# Imports

In [ ]:
import numpy as np
from numba import njit

import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']

import os 
import time

import aespm as ae

# Load data (example)

## FD curves

### Pre-processing raw data

In [ ]:
# Helper functions to load FD curves
def read_fd(obj, width=None, factor=None):
    index = np.argmax(obj.data[0])
    if width is None:
        width = np.min([index, len(obj.data[0])-index])
    fwd = np.zeros([3, width])
    bkd = np.zeros([3, width])
    for i in range(3):
        fwd[i] = obj.data[i][index-width:index]
        bkd[i] = obj.data[i][index:index+width]
    if factor is not None:
        fwd[1] /= factor
        bkd[1] /= factor
    return fwd, bkd        

def find_all_sign_change_indices(arr):
    signs = np.sign(arr)  # Get sign of elements (-1, 0, or 1)
    changes = np.where(signs[:-1] != signs[1:])[0]  # Compare consecutive elements
    return changes
    
def find_thres(ph, amp, thres=80):
    ph_offset = ph - ph[0] + 90
    index = find_all_sign_change_indices(ph_offset-thres)
    if len(index) == 1:
        return amp[index][0]
    elif len(index) > 1:
        return amp[index[0]]
    else:
        return np.nan
    
def conver_unit(p, limits, reverse=False):
    '''
    Project  p from (0, 1) to (p_min, p_max).
    '''
    out = np.zeros_like(p)
    for i in range(len(p)):
        if reverse is False:
            p_min, p_max = limits[i]
            out[i] = p_min + (p_max - p_min) * p[i]
        else:
            p_min, p_max = limits[i]
            out[i] = (p[i] - p_min) / (p_max - p_min)
    return out

In [ ]:
folder = os.path.join('data', '250315', 'pickles')

fname = '250315_Cali1_MOBO training.pickle'

save = '250315_Cali1_MOBO'

import pickle

with open(os.path.join(folder, fname), 'rb') as fopen:
    param = pickle.load(fopen)

train_obj = param['y_measured_tensor']

# visualize the topograph
import aespm as ae

map_name = 'Cali_res1_0001.ibw'

folder = os.path.join('data', '250315', 'CaliSample01')

save = '250315_Cali_MOBO_01'


# import pickle

topo = ae.tools.load_ibw(os.path.join(folder, map_name))

factor = param['factor'] * topo.header['InvOLS'] * 1e9

In [ ]:
folder = os.path.join('data', '250315', 'pickles')


with open(os.path.join(folder, '250315_Cali1_MOBO.pickle'), 'rb') as fopen:
    param_clean = pickle.load(fopen)

param_clean.keys()

In [ ]:
# Load FD curves from experiment

folder = os.path.join('data', '250315', 'CaliSample01')

data = []
files = []

for file in os.listdir(folder):
    if file.startswith('Cali_FD'):
        files.append(os.path.join(folder, file))

files = sorted(files)

for file in files:
    data.append(ae.tools.load_ibw(file))

In [ ]:
fwds = []
bkds = []
drives = []
thresholds = []

x = []

for i in range(len(data)):
    temp = data[i]
    drive = temp.header['DriveAmplitude']
    drives.append(drive)
    fwd, bkd = read_fd(temp, factor=1e-9)
    ending = np.min(fwd[1])
    setpoint = ending * 4
    fwds.append(fwd)
    bkds.append(bkd)
    amp_thres = find_thres(fwd[2], fwd[1], thres=85)
    thresholds.append(amp_thres / setpoint * 1e2)
    x.append(fwd[1] / setpoint * 1e2)

In [ ]:
tp = fwds[0]

L = 50

height = np.zeros([len(fwds), L])
amp = np.zeros([len(fwds), L])
phase = np.zeros([len(fwds), L])

for i in range(len(fwds)):
    tp = fwds[i]
    height[i] = (tp[0] - np.min(tp[0][-L:]))[-L:]*1e9
    amp[i] = tp[1][-L:][::-1]
    phase[i] = tp[2][-L:][::-1]


fig, ax = plt.subplots(1,2,figsize=[10,3])
for i in range(len(fwds)):
    ax[0].plot(height[i], amp[i])
ax[0].set_xlabel('Height (nm)')
ax[0].set_ylabel('Amplitude (nm)')

for i in range(len(fwds)):
    ax[1].plot(height[i], phase[i])
ax[1].set_xlabel('Height (nm)')
ax[1].set_ylabel('Phase (degrees)')

plt.tight_layout()

# plt.figure()
# for i in range(len(fwds)):
#     plt.plot(height[i], amp[i])
# plt.xlabel('Height (nm)')
# plt.ylabel('Amplitude (nm)')

# plt.figure()
# for i in range(len(fwds)):
#     plt.plot(height[i], phase[i])
# plt.xlabel('Height (nm)')
# plt.ylabel('Phase (degrees)')

In [ ]:
import aespm.tools as at

folder1 = os.path.join('data', '260514', 'Multi75')
folder2 = os.path.join('data', '260514', 'Tap300')

data = []

for file in sorted(os.listdir(folder2)):
    if file.startswith('AlScN_FD'):
        data.append(at.load_ibw(os.path.join(folder2, file)))

print(len(data))

In [ ]:
header = data[0].header
header

In [ ]:
tp = data[0]
plt.plot(tp.data[0], tp.data[2])


In [ ]:
fwds = []
bkds = []
drives = []
thresholds = []
drives_exp = []

x = []

for i in range(len(data)):
    temp = data[i]
    drive = temp.header['DriveAmplitude']
    drives.append(drive)
    fwd, bkd = read_fd(temp, factor=1e-9)
    # ending = np.min(fwd[1])
    # setpoint = ending * 10
    amp_free = drive * 420 / 3
    fwds.append(fwd)
    bkds.append(bkd)
    drives_exp.append(np.mean([fwd[0], bkd[0]]))
    
    x.append(fwd[0] / amp_free * 1e2)

factor1 = temp.header['AmpInvOLS'] * 1e9 * 420 / 3

In [ ]:
# tp = fwds[0]

L = 1000

N = len(fwds) - 1

height1 = np.zeros([N, L])
amp1 = np.zeros([N, L])
phase1 = np.zeros([N, L])

height2 = np.zeros([N, L])
amp2 = np.zeros([N, L])
phase2 = np.zeros([N, L])

for i in range(N):
    tp = fwds[i]
    height1[i] = (tp[0] - np.min(tp[0][-L:]))[-L:]*1e9
    amp1[i] = tp[1][-L:][::-1]
    phase1[i] = tp[2][-L:][::-1]

    tp = bkds[i]
    height2[i] = (tp[0] - np.min(tp[0][:L]))[:L]*1e9
    amp2[i] = tp[1][:L][::-1]
    phase2[i] = tp[2][:L][::-1]


fig, ax = plt.subplots(1,2,figsize=[10,3])
for i in range(N):
    ax[0].plot(height1[i], amp1[i])
ax[0].set_xlabel('Height (nm)')
ax[0].set_ylabel('Amplitude (nm)')

for i in range(N):
    ax[1].plot(height1[i], phase1[i], '.-')
ax[1].set_xlabel('Height (nm)')
ax[1].set_ylabel('Phase (degrees)')

plt.tight_layout()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

fig, ax = plt.subplots(1, 2, figsize=[8,3])

drive = np.array(drives) * factor1
# drive = (np.array(drives_exp)-np.min(drives_exp)) * 1e9

N = len(drive) - 1

# cmap = plt.get_cmap('RdPu')
# cmap = plt.get_cmap('turbo')
cmap = plt.get_cmap('rainbow')


norm = mpl.colors.Normalize(vmin=np.min(drive), vmax=np.max(drive))

for i in range(N):
    ax[0].plot(
        height1[i],
        amp1[i], '-', alpha=1, lw=1.,
        color=cmap(norm(drive[i]))
    )
    ax[1].plot(
        height1[i],
        phase1[i], '-', alpha=1, lw=1.,
        color=cmap(norm(drive[i]))
    )

    # ax[0].plot(
    #     height2[i],
    #     amp2[i], '-', alpha=1, lw=1.,
    #     color=cmap(norm(drive[i]))
    # )
    # ax[1].plot(
    #     height2[i],
    #     phase2[i], '-', alpha=1, lw=1.,
    #     color=cmap(norm(drive[i]))
    # )

# ax.set_xticks([340, 350, 360, 370, 380, 390])
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

cbar = fig.colorbar(sm, ax=ax[1])
cbar.set_label(r"Drive amplitude (nm)")

# ax.set_yscale('log')

ax[0].set_xlabel('Distance (nm)')
ax[0].set_ylabel('Amplitude (nm)')
ax[1].set_xlabel('Distance (nm)')
ax[1].set_ylabel('Phase (degree)')

fig.tight_layout()

plt.savefig('output/different tunes amp phase horizontal tap300 fwd.png', dpi=400, pad_inches=0.1)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

fig, ax = plt.subplots(1, 2, figsize=[10,4])

drive = np.array(drives) * factor1
# drive = (np.array(drives_exp)-np.min(drives_exp)) * 1e9

N = len(drive) - 1

# cmap = plt.get_cmap('RdPu')
# cmap = plt.get_cmap('turbo')
cmap = plt.get_cmap('rainbow')


norm = mpl.colors.Normalize(vmin=np.min(drive), vmax=np.max(drive))

for i in range(N):
    # ax[0].plot(
    #     height1[i],
    #     amp1[i], '-', alpha=1, lw=1.,
    #     color=cmap(norm(drive[i]))
    # )
    # ax[1].plot(
    #     height1[i],
    #     phase1[i], '-', alpha=1, lw=1.,
    #     color=cmap(norm(drive[i]))
    # )

    ax[0].plot(
        height2[i],
        amp2[i], '-', alpha=1, lw=1.,
        color=cmap(norm(drive[i]))
    )
    ax[1].plot(
        height2[i],
        phase2[i], '-', alpha=1, lw=1.,
        color=cmap(norm(drive[i]))
    )

# ax.set_xticks([340, 350, 360, 370, 380, 390])
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

cbar = fig.colorbar(sm, ax=ax[1])
cbar.set_label(r"Drive amplitude (nm)")

# ax.set_yscale('log')

ax[0].set_xlabel('Distance (nm)')
ax[0].set_ylabel('Amplitude (nm)')
ax[1].set_xlabel('Distance (nm)')
ax[1].set_ylabel('Phase (degree)')

fig.tight_layout()

plt.savefig('output/different tunes amp phase horizontal tap300 bkd.png', dpi=400, pad_inches=0.1)

In [ ]:
# np.savez('output/Multi75_AlScN.npz', drive=drives_si, amp=amp1, height=height1, phase=phase1)

### Load saved data

In [ ]:
# data = np.load('output/cali_fd.npz')
data = np.load('output/Multi75_AlScN.npz')

height = data['height'][:,::5]
amp = data['amp'][:,::5]
phase = data['phase'][:,::5]
drive = data['drive']

idx = np.argsort(drive)
drive = drive[idx]
height = height[idx]
amp = amp[idx]
phase = phase[idx]

## Scan lines with parameters

### Pre-processing raw data

### Load saved data

In [ ]:
# Load all the FD curves

# data = np.load('output/cali_fd.npz')
data = np.load('output/Tap300_AlScN.npz')
# data = np.load('output/Multi75_AlScN.npz')

height = data['height']
phase = data['phase']
amp = data['amp']
drive = data['drive'][:-1]

# plt.plot(amp[0])

drive_fd = np.zeros(len(drive))

for i in range(len(drive)):
    drive_fd[i] = np.mean(amp[i, -10:])

# data = np.load('output/cleaned_traces.npz')
# print(data.keys())

# traces = data['traces']
# x_measured = data['x_measured']
# y_measured = data['y_measured']
# topo = data['topo']

# drive

In [ ]:
map_name = 'Image0002.ibw'

# Topo file
topo = ae.tools.load_ibw(os.path.join('data', map_name))

# control parameter grid
data = np.load(os.path.join('data', '260507-300kHz-AlScN.npz'))
print(data.keys())

In [ ]:
# np.savez('output/260507-300kHz-AlScN-topo.npz', data=topo.data, header=topo.header)

In [ ]:
drive = data['drives'] * (258.8 / 2.14) * topo.header['AmpInvOLS'] *1e9
drive

In [ ]:
drive_fd

In [ ]:
data['data'].shape

In [ ]:
drive_exp = data['drives'] * (258.8 / 2.14) * topo.header['AmpInvOLS'] *1e9
setpoint_exp = data['setpoints']
igain_exp = data['i_gain'][:-2]
scan_rate = data['scan_rate']

n_speed = 5
n_drive = 11
n_setpoint = 8
n_gain = 5

traces = data['data']

traces_height = np.zeros([5, 11, 8, 5, 2, 256])

# # data['data'].shape = (5, 11, 8, 5, 3, 8, 256)

# for i in range(n_speed):
#     for j in range(n_drive):
#         for k in range(n_setpoint):
#             for l in range(n_gain):
#                 for m in range(2):
#                     traces_exp[i,j,k,l,m] = (traces[i,j,k,l,-1,m]-np.min(traces[i,j,k,l,-1,m]))*1e9

h = traces[:, :, :, :-2, 1, :2, :]   # shape: (5, 11, 8, 5, 2, 256)

traces_exp = (h - h.min(axis=-1, keepdims=True)) * 1e9

traces_exp.shape

In [ ]:
igain_exp

## Generate realistic scans with the same height change rate

In [ ]:
tp = topo.data[0]
tp = (tp - np.min(tp)) * 1e9

In [ ]:
# Let's generate the distribution of the height change rate based on a full topo
plt.imshow(tp)
plt.colorbar()

In [ ]:
topo_diff = np.diff(tp)
plt.imshow(topo_diff)

In [ ]:
plt.hist(topo_diff, bins=20)

In [ ]:
from codes.synthetic_scan_line_generator import (
    generate_synthetic_scan_line,
    plot_generated_line_and_rate,
    toy_trace_retrace_scan,
    plot_simulated_scan,
)

# Replace this with your measured full map.
# Shape: (ny, nx)
h_map = tp

pack = generate_synthetic_scan_line(
    h_map,
    n_pixels=256,
    dx_hat=1.0,
    scan_speed_hat=1e3,
    mode="filtered_iid",
    direction="x",
    corr_sigma_px=1.0,
    random_state=35,
)

h_line = pack["generated"]["h_hat"]


print(pack["comparison"])

plot_generated_line_and_rate(
    h_map,
    h_line,
    pack["rate_dist"],
    dx_hat=1.0,
    scan_speed_hat=1e3,
)

In [ ]:
# # from codes.Scanner_fixed import *
# from codes.Scanner_numba_substeps_v3 import *

In [ ]:
# drives_si = drive_fd
# measured_fd = {}


# # Assume these are lists/arrays where index i corresponds to drives[i]
# # z_arrays[i], A_arrays[i], phi_arrays[i]
# # each could be a 1D FD curve, a scan line, or an image
# measured_fd = {
#     float(drive): {
#         "d": height[i],
#         "A": amp[i],
#         "phi": phase[i],
#     }
#     for i, drive in enumerate(drives_si)
# }


# measured_fd_ext, zero_amp_diag = extend_measured_fd_dict_linear_to_zero_amplitude(
#     measured_fd,
#     n_fit=5,
#     n_bridge=50,
#     d_zero=None,              # infer A=0 location from near-contact slope
#     phi_mode="linear",
#     F_mode="nearest",
#     max_back_extrap_factor=5.0,
# )

# fd_surface = FDDriveSurface.from_measured_fd(
#     measured_fd_ext,
#     x_mode="d_over_A0",
#     common_range="intersection",
#     n_x=1000,
#     far_field_frac=0.10,
#     far_field_side="largest_d",
#     drive_key_is_A0=False,
# )

# conv_L = 1.0  # or scanner.conversion["A"] if you are using normalized units


# fd_model = make_normalized_fd_lookup(
#     fd_surface,
#     conv_L=1.0,
#     conv_A=1.0,
#     conv_F=1.0,
# )



# params = {
#     "k": 25,
#     "A": float(np.nanmedian(amp)),   # fallback only
#     "d0": float(np.nanmax(height) * 0.8),
#     "R": 10,
#     "H": 1e-19,
#     "E_star": 1e9,
#     "Q": 250,
# }

# conversions = {k: 1.0 for k in params}

# scanner = ScannerFD(
#     params=params,
#     conversions=conversions,
#     fd_model=fd_model,
# )

# print("A, phi, F at drive 2:", scanner.measure_from_fd(5.0, drive_nm=20.0))
# print("A, phi, F at drive 8:", scanner.measure_from_fd(5.0, drive_nm=80.0))
# print("A0 at drive 2:", scanner.get_A0_hat(2.0))
# print("A0 at drive 8:", scanner.get_A0_hat(8.0))
# print("FD range at drive 2:", scanner.distance_range(2.0))

In [ ]:
import numpy as np
import importlib

import codes.Scanner_fixed_extended_fd as sfe
import codes.Scanner_numba_substeps_v3 as sn

importlib.reload(sfe)
importlib.reload(sn)

# Explicitly patch the ScannerFD class from Scanner_fixed_extended_fd
sn.install_numba_substep_methods(sfe.ScannerFD, verbose=True)

FDDriveSurface = sfe.FDDriveSurface
make_normalized_fd_lookup = sfe.make_normalized_fd_lookup
ScannerFD = sfe.ScannerFD

In [ ]:
# Load all the FD curves

# data = np.load('output/cali_fd.npz')
data = np.load('output/Tap300_AlScN.npz')
# data = np.load('output/Multi75_AlScN.npz')

height = data['height']
phase = data['phase']
amp = data['amp']
drive = data['drive'][:-1]

# plt.plot(amp[0])

drive_fd = np.zeros(len(drive))

for i in range(len(drive)):
    drive_fd[i] = np.mean(amp[i, -10:])


# # from codes.Scanner_fixed import *
# from codes.Scanner_numba_substeps_v3 import *
# from codes.Scanner_fixed_extended_fd import *

drives_si = drive_fd * 0.8
measured_fd = {}

measured_fd = {
    float(drive): {
        "d": height[i],
        "A": amp[i],
        "phi": phase[i],
    }
    for i, drive in enumerate(drives_si)
}
conv_L = 1

params = {
    "k": 25,
    "A": float(np.nanmedian(amp)),   # fallback only
    "d0": float(np.nanmax(height) * 0.8),
    "R": 10,
    "H": 1e-19,
    "E_star": 1e9,
    "Q": 250,
}

conversions = {k: 1.0 for k in params}

from codes.Scanner_fixed_extended_fd import (
    FDDriveSurface,
    make_normalized_fd_lookup,
    ScannerFD,
)

surface = FDDriveSurface.from_measured_fd(
    measured_fd,
    x_mode="d_over_A0",
    common_range="union",
    n_x=1024,
    extend_to_zero_amplitude=True,
    extension_n_fit=5,
    extension_n_bridge=30,
    extension_phi_mode="linear",
    extension_F_mode="nearest",
)

fd_model = make_normalized_fd_lookup(
    surface,
    conv_L=conv_L,
)

scanner = ScannerFD(
    params=params,
    conversions=conversions,
    fd_model=fd_model,
)

print("A, phi, F at drive 2:", scanner.measure_from_fd(5.0, drive_nm=20.0))
print("A, phi, F at drive 8:", scanner.measure_from_fd(5.0, drive_nm=80.0))
print("A0 at drive 2:", scanner.get_A0_hat(2.0))
print("A0 at drive 8:", scanner.get_A0_hat(8.0))
print("FD range at drive 2:", scanner.distance_range(2.0))

In [ ]:
h_truth = h_line

In [ ]:
# fd20 = scanner.prepare_fd_table_for_drive(drive_nm=10, n_d=4096)

out = scanner.simulate_trace_retrace_substeps_fast(
    h_truth,
    drive_nm=10,
    setpoint=0.6,
    P=0.28*50,
    I=0.48*50,
    scan_speed_hat=1.0,
    dx_hat=1.0,
    n_substeps=50,
    fd_table=None,
    h_smooth_sigma_px=0.0,
    d_init_hat=0,
    A_meas_init_hat=1,
)

fig, ax = plt.subplots(1,3,figsize=[10,3.])
ax[0].plot(out['trace']['d_hat'])
ax[0].plot(out['retrace']['d_hat'])
ax[2].plot(out['trace']['phi_deg'])
ax[2].plot(out['retrace']['phi_deg'])
ax[1].plot(out['trace']['A_hat'])
ax[1].plot(out['retrace']['A_hat'])
ax[0].set_ylabel('Height (nm)')
ax[1].set_ylabel('Amplitude (nm)')
ax[2].set_ylabel('Phase (degree)')
ax[0].set_title('Drive=20 nm, setpoint=0.6')
ax[1].set_title('P=0.28, I=0.48')

plt.tight_layout()

### Static calibration 

In [ ]:
traces_exp.shape

In [ ]:
# Generate 8 different scan lines with the same signature as the measured topo

seeds = np.random.randint(0, 256, 8)

h_line_simu = np.zeros([8, 256])
for i in range(8):
    pack = generate_synthetic_scan_line(
        h_map,
        n_pixels=256,
        dx_hat=1.0,
        scan_speed_hat=1,
        mode="filtered_iid",
        direction="x",
        corr_sigma_px=1.0,
        random_state=seeds[i],
    )
    
    h_line = pack["generated"]["h_hat"]
    h_line_simu[i] = h_line



In [ ]:
# Alternatively, we can get 8 lines from the ground truth directly for comparison

h_line_exp = np.zeros([8, 256])
for i in range(8):
    h_line_exp[i] = h_map[seeds[i]]

In [ ]:
from codes.synthetic_scan_line_generator_v2 import generate_synthetic_scan_line
from codes.dt_static_dynamic_calibration_scaffold import make_synthetic_lines_from_map

synthetic_lines, line_packs = make_synthetic_lines_from_map(
    h_map,
    generate_synthetic_scan_line=generate_synthetic_scan_line,
    n_lines=8,
    n_pixels=256,
    dx_hat=1.0,

    # This value only matters for reported rate / max-rate constraints.
    # The actual scanner speed is passed later during simulation.
    scan_speed_hat_for_generation=1.0,

    mode="filtered_iid",
    direction="x",
    corr_sigma_px=1.0,

    preserve_max_abs_dh=True,
    slope_limit_mode="rescale",
    seed0=35,
)

In [ ]:
from codes.dt_trace_classifier import TraceUsabilityConfig, classify_height_trace_pair

usability_cfg = TraceUsabilityConfig(
    trim=10,
    min_mad=0.02,        # nm, adjust to your noise floor
    min_range=0.10,      # nm
    min_grad_mad=0.002,  # nm / pixel
)

cond = (2, 8, 5, 2)
si, di, spi, gi = cond

info = classify_height_trace_pair(
    traces_exp[si, di, spi, gi, 0],
    traces_exp[si, di, spi, gi, 1],
    cfg=usability_cfg,
)

print(info)

In [ ]:
synthetic_lines[0].shape

In [ ]:
from codes.dt_static_dynamic_calibration_scaffold_v3 import ScannerRunConfig
from codes.dt_gp_static_calibration_v3 import (
    GPSubsetCalibrationConfig,
    fit_gp_static_controller_calibration,
    predict_PI_for_full_grid,
)
from codes.dt_trace_classifier import (
    TraceUsabilityConfig,
    classify_height_trace_pair,
)

import importlib
import codes.dt_gp_static_calibration_v3 as gpcal

importlib.reload(gpcal)

GPSubsetCalibrationConfig = gpcal.GPSubsetCalibrationConfig
fit_gp_static_controller_calibration = gpcal.fit_gp_static_controller_calibration
predict_PI_for_full_grid = gpcal.predict_PI_for_full_grid

scanner_cfg = ScannerRunConfig(
    dx_hat=1.0,
    n_substeps=50,
    tau_A=None,
    tau_phi=None,
    tau_z=None,
    z_rate_limit_hat=1e6,
    d_init_hat=0.0,
    A_meas_init_hat=1.0,
    phi_meas_init_deg=120.0,
    T_I=3e-1,
    h_smooth_sigma_px=0.0,
    fd_n_d=4096,
    use_fast=True,
)

gp_cfg = GPSubsetCalibrationConfig(
    n_train=200,
    n_test=50,
    seed=0,

    n_lines_train=3,
    n_lines_test=8,

    trim=10,

    # Classifier thresholds.
    filter_uninformative=True,
    min_mad=0.02,
    min_range=0.10,
    min_grad_mad=0.002,
    min_nonzero_frac=0.05,

    # Local P/I fit.
    bounds_logPI=(
        (-3, 3),   # log10 P
        (-3, 3),   # log10 I
    ),
    use_grid_init=True,
    n_grid_P=21,
    n_grid_I=21,
    do_local_refine=True,
    maxiter_local=200,
    w_trrt = 100.0,
    w_grad_mean = 10.0,
    w_grad_std = 5.0,
    w_grad_p95 = 0,
    w_height_mad = 0,
    w_height_range = 0,
)

gp_fit = fit_gp_static_controller_calibration(
    scanner,
    traces_exp,
    # synthetic_lines,
    h_line_exp,
    scan_speeds_hat=scan_rate,
    drives_nm=drive_exp,
    setpoints=setpoint_exp,
    i_gains_exp=igain_exp,
    scanner_cfg=scanner_cfg,
    cfg=gp_cfg,
    verbose=True,
)

print(gp_fit["test_summary"])
print("Rejected:", len(gp_fit["rejected_conditions"]))

In [ ]:
from codes.dt_static_fit_visualization import (
    test_rows_to_dataframe,
    plot_static_fit_dashboard,
    plot_metric_alignment,
    plot_test_scan_examples,
)

In [ ]:
df_test = test_rows_to_dataframe(gp_fit)
df_test.head()

In [ ]:
figs = plot_static_fit_dashboard(
    gp_fit,
    metric_keys=[
        "trrt",
        "grad_mean",
        "grad_std",
        "grad_p95",
        "height_mad",
        "height_range_p99_p01",
    ],
    log_metric_alignment=False,
)

In [ ]:
fig, axes = plot_metric_alignment(
    gp_fit,
    metric_keys=[
        "trrt",
        "grad_mean",
        "grad_std",
        "grad_p95",
        "height_mad",
        "height_range_p99_p01",
    ],
    log_scale=False,
)

In [ ]:
figs, selected_rows = plot_test_scan_examples(
    scanner,
    traces_exp,
    # synthetic_lines,
    h_line_exp,
    gp_fit,

    scan_speeds_hat=scan_rate,
    drives_nm=drive_exp,
    setpoints=setpoint_exp,
    i_gains_exp=igain_exp,

    scanner_cfg=scanner_cfg,

    n_examples=6,
    mode="mixed",          # "best", "worst", "random", or "mixed"
    synthetic_line_index=0,
    trim=10,
    preprocess="center",   # "center", "raw", or "robust_zscore"
)

In [ ]:
df_test.sort_values("test_loss", ascending=False).head(10)[
    [
        "cond",
        "test_loss",
        "P_pred",
        "I_pred",
        "exp_trrt",
        "sim_trrt",
        "exp_grad_mean",
        "sim_grad_mean",
        "exp_grad_std",
        "sim_grad_std",
        "exp_height_mad",
        "sim_height_mad",
    ]
]

### Compare the static and dynamics FD response

In [ ]:
traces_exp.shape

In [ ]:
plt.plot(traces_exp[2, 8, 5, 4, 1])

### Dynamics calibration and prediction

In [ ]:
from codes.dt_static_dynamic_calibration_scaffold import (
    build_dynamic_residual_dataset,
    fit_quality_safety_models_from_rows,
)

rows = build_dynamic_residual_dataset(
    scanner,
    traces_exp_height=traces_exp,
    traces_exp_amp=traces_amp_exp,        # same shape, or None
    traces_exp_phase=traces_phase_exp,    # same shape, or None
    synthetic_lines=synthetic_lines,
    scan_speeds_hat=scan_speeds_hat,
    drives_nm=drives_nm,
    setpoints=setpoints,
    i_gains_exp=i_gains_exp,

    static_pars=static_fit["best_pars"],
    scanner_cfg=scanner_cfg,
    trim=10,
    n_lines=8,
)

model_pack = fit_quality_safety_models_from_rows(rows)

## Clibration grating sample

In [ ]:
# Load all the FD curves

data = np.load('output/cali_fd.npz')
# data = np.load('output/Tap300_AlScN.npz')
# data = np.load('output/Multi75_AlScN.npz')

height = data['height']
phase = data['phase']
amp = data['amp']
drive = data['drive']

# plt.plot(amp[0])

drive_fd = np.zeros(len(drive))

for i in range(len(drive)):
    drive_fd[i] = np.mean(amp[i, -10:])

# data = np.load('output/cleaned_traces.npz')
# print(data.keys())

# traces = data['traces']
# x_measured = data['x_measured']
# y_measured = data['y_measured']
# topo = data['topo']

# drive

In [ ]:
drives_si = drive_exp
measured_fd = {}


# Assume these are lists/arrays where index i corresponds to drives[i]
# z_arrays[i], A_arrays[i], phi_arrays[i]
# each could be a 1D FD curve, a scan line, or an image
measured_fd = {
    float(drive): {
        "d": height[i],
        "A": amp[i],
        "phi": phase[i],
    }
    for i, drive in enumerate(drives_si)
}


measured_fd_ext, zero_amp_diag = extend_measured_fd_dict_linear_to_zero_amplitude(
    measured_fd,
    n_fit=5,
    n_bridge=50,
    d_zero=None,              # infer A=0 location from near-contact slope
    phi_mode="linear",
    F_mode="nearest",
    max_back_extrap_factor=5.0,
)

fd_surface = FDDriveSurface.from_measured_fd(
    measured_fd_ext,
    x_mode="d_over_A0",
    common_range="intersection",
    n_x=1000,
    far_field_frac=0.10,
    far_field_side="largest_d",
    drive_key_is_A0=False,
)

conv_L = 1.0  # or scanner.conversion["A"] if you are using normalized units

fd_model = make_normalized_fd_lookup(
    fd_surface,
    conv_L=1.0,
    conv_A=1.0,
    conv_F=1.0,
)



params = {
    "k": 25,
    "A": float(np.nanmedian(amp)),   # fallback only
    "d0": float(np.nanmax(height) * 0.8),
    "R": 10,
    "H": 1e-19,
    "E_star": 1e9,
    "Q": 250,
}

conversions = {k: 1.0 for k in params}

scanner = ScannerFD(
    params=params,
    conversions=conversions,
    fd_model=fd_model,
)


print("A, phi, F at drive 20:", scanner.measure_from_fd(50.0, drive_nm=20.0))
print("A, phi, F at drive 80:", scanner.measure_from_fd(50.0, drive_nm=80.0))
print("A0 at drive 20:", scanner.get_A0_hat(20.0))
print("A0 at drive 80:", scanner.get_A0_hat(80.0))
print("FD range at drive 20:", scanner.distance_range(20.0))

In [ ]:
import numpy as np
import os

file1 = os.path.join('data', 'AR_Height_AE.npz')

data1 = np.load(file1)

print('scan_rate:', data1['scan_rate'],)
print('v_ac:', data1['v_ac'],)
print('setpoint:', data1['setpoint'],)
print('i_gain:', data1['i_gain'])
print('data:', data1['data'].shape)

# # Outputs:

# scan_rate: [0.25 0.5  0.75 1.   2.   3.   4.   5.   6.  ]
# v_ac: [0.15   0.1375 0.125  0.1125 0.1    0.0875 0.075  0.0625 0.05   0.0375
#  0.025 ]
# setpoint: [4.2 4.1 4.  3.9 3.8 3.7 3.6 3.5 3.4 3.3 3.2 3.1 3.  2.9 2.8 2.7]
# i_gain: [ 10.  25.  40.  55.  70.  85. 100. 115. 130. 145. 160.]
# data: (9, 11, 16, 11, 8, 512)

In [ ]:
drives_exp = np.linspace(3.91, 23.47, num=11)
setpoints_exp = np.linspace(0.8, 0.05, num=16)
i_gain_exp = np.array([ 10.,  25.,  40.,  55.,  70.,  85., 100., 115., 130., 145., 160.])
scan_rate_exp = np.array([0.25, 0.5,  0.75, 1.,   2.,   3.,   4.,   5.,  6.  ])
traces = data1['data']
traces.shape

In [ ]:
h = traces[:, :, :, :, :2, ::2]   # shape: (5, 11, 8, 5, 2, 256)

traces_exp = (h - h.min(axis=-1, keepdims=True)) * 1e9

traces_exp.shape

In [ ]:
traces_amp_exp = traces[:, :, :, :, 2:4, ::2] * 1e9
traces_phase_exp = traces[:, :, :, :, 4:6, ::2]
plt.plot(traces_amp_exp[1,3,5,0,0])

In [ ]:
t = np.load('output/cleaned_traces.npz')

topo = t['topo']

In [ ]:
t['topo'].shape

In [ ]:
# Generate 8 different scan lines with the same signature as the measured topo

seeds = np.random.randint(0, 256, 8)
h_map = (topo[0] - np.min(topo[0])) * 1e9

h_line_simu = np.zeros([8, 256])
for i in range(8):
    pack = generate_synthetic_scan_line(
        h_map,
        n_pixels=256,
        dx_hat=1.0,
        scan_speed_hat=1e3,
        mode="filtered_iid",
        direction="x",
        corr_sigma_px=1.0,
        random_state=seeds[i],
    )
    
    h_line = pack["generated"]["h_hat"]
    h_line_simu[i] = h_line



In [ ]:
# Alternatively, we can get 8 lines from the ground truth directly for comparison

h_line_exp = np.zeros([8, 256])
for i in range(8):
    h_line_exp[i] = h_map[seeds[i]]

In [ ]:
from codes.synthetic_scan_line_generator_v2 import generate_synthetic_scan_line
from codes.dt_static_dynamic_calibration_scaffold import make_synthetic_lines_from_map

synthetic_lines, line_packs = make_synthetic_lines_from_map(
    h_map,
    generate_synthetic_scan_line=generate_synthetic_scan_line,
    n_lines=8,
    n_pixels=256,
    dx_hat=100.0,

    # This value only matters for reported rate / max-rate constraints.
    # The actual scanner speed is passed later during simulation.
    scan_speed_hat_for_generation=1.0,

    mode="filtered_iid",
    direction="x",
    corr_sigma_px=1.0,

    preserve_max_abs_dh=True,
    slope_limit_mode="rescale",
    seed0=35,
)

In [ ]:
from codes.dt_trace_classifier import TraceUsabilityConfig, classify_height_trace_pair

usability_cfg = TraceUsabilityConfig(
    trim=10,
    min_mad=0.02,        # nm, adjust to your noise floor
    min_range=0.10,      # nm
    min_grad_mad=0.002,  # nm / pixel
)

cond = (2, 8, 5, 4)
si, di, spi, gi = cond

info = classify_height_trace_pair(
    traces_exp[si, di, spi, gi, 0],
    traces_exp[si, di, spi, gi, 1],
    cfg=usability_cfg,
)

print(info)

In [ ]:
traces_exp.shape

In [ ]:
from codes.dt_static_dynamic_calibration_scaffold_v3 import ScannerRunConfig
from codes.dt_gp_static_calibration_v3 import (
    GPSubsetCalibrationConfig,
    fit_gp_static_controller_calibration,
    predict_PI_for_full_grid,
)
from codes.dt_trace_classifier import (
    TraceUsabilityConfig,
    classify_height_trace_pair,
)

import importlib
import codes.dt_gp_static_calibration_v3 as gpcal

importlib.reload(gpcal)

GPSubsetCalibrationConfig = gpcal.GPSubsetCalibrationConfig
fit_gp_static_controller_calibration = gpcal.fit_gp_static_controller_calibration
predict_PI_for_full_grid = gpcal.predict_PI_for_full_grid

scanner_cfg = ScannerRunConfig(
    dx_hat=1.0,
    n_substeps=50,
    tau_A=None,
    tau_phi=None,
    tau_z=None,
    z_rate_limit_hat=1e6,
    d_init_hat=100.0,
    A_meas_init_hat=1.0,
    phi_meas_init_deg=120.0,
    T_I=3e-1,
    h_smooth_sigma_px=0.0,
    fd_n_d=4096,
    use_fast=True,
)

gp_cfg = GPSubsetCalibrationConfig(
    n_train=200,
    n_test=50,
    seed=0,

    n_lines_train=3,
    n_lines_test=8,

    trim=10,

    # Classifier thresholds.
    filter_uninformative=True,
    min_mad=0.02,
    min_range=0.10,
    min_grad_mad=0.002,
    min_nonzero_frac=0.05,

    # Local P/I fit.
    bounds_logPI=(
        (-3, 3),   # log10 P
        (-3, 3),   # log10 I
    ),
    use_grid_init=True,
    n_grid_P=21,
    n_grid_I=21,
    do_local_refine=True,
    maxiter_local=200,
    w_trrt = 5.0,
    w_grad_mean = 1.0,
    w_grad_std = 1.0,
    w_grad_p95 = 0,
    w_height_mad = 0,
    w_height_range = 0,
)

gp_fit = fit_gp_static_controller_calibration(
    scanner,
    traces_exp,
    # synthetic_lines,
    h_line_exp,
    scan_speeds_hat=scan_rate_exp,
    drives_nm=drives_exp,
    setpoints=setpoints_exp,
    i_gains_exp=i_gain_exp,
    scanner_cfg=scanner_cfg,
    cfg=gp_cfg,
    verbose=True,
)

# drives_exp = np.linspace(3.91, 23.47, num=11)
# setpoints_exp = np.linspace(0.8, 0.05, num=16)
# i_gain_exp = np.array([ 10.,  25.,  40.,  55.,  70.,  85., 100., 115., 130., 145., 160.])
# scan_rate_exp = np.array([0.25, 0.5,  0.75, 1.,   2.,   3.,   4.,   5.,  6.  ])
# traces = data1['data']
# traces.shape

print(gp_fit["test_summary"])
print("Rejected:", len(gp_fit["rejected_conditions"]))

In [ ]:
df_test = test_rows_to_dataframe(gp_fit)
df_test.head()

In [ ]:
figs = plot_static_fit_dashboard(
    gp_fit,
    metric_keys=[
        "trrt",
        "grad_mean",
        "grad_std",
        "grad_p95",
        "height_mad",
        "height_range_p99_p01",
    ],
    log_metric_alignment=False,
)

In [ ]:
figs, selected_rows = plot_test_scan_examples(
    scanner,
    traces_exp,
    # synthetic_lines,
    h_line_exp,
    gp_fit,

    scan_speeds_hat=scan_rate_exp,
    drives_nm=drives_exp,
    setpoints=setpoints_exp,
    i_gains_exp=i_gain_exp,

    scanner_cfg=scanner_cfg,

    n_examples=6,
    mode="mixed",          # "best", "worst", "random", or "mixed"
    synthetic_line_index=0,
    trim=10,
    preprocess="center",   # "center", "raw", or "robust_zscore"
)

In [ ]:
import numpy as np


class RoughnessRateMemory:
    """
    Online memory of recent scan-line roughness.

    Stores height-change-rate arrays:

        rate = scan_speed_hat * diff(h) / dx_hat

    and generates synthetic scan lines by sampling from a recency-weighted
    distribution of past rate arrays.
    """

    def __init__(
        self,
        memory_size=20,
        half_life_lines=5.0,
        dx_hat=1.0,
        center_height=True,
        random_state=None,
    ):
        self.memory_size = int(memory_size)
        self.half_life_lines = float(half_life_lines)
        self.dx_hat = float(dx_hat)
        self.center_height = bool(center_height)
        self.rng = np.random.default_rng(random_state)

        self.rate_lines = []
        self.meta = []

    def _decay_weights(self):
        """
        Newest line has largest weight.

        rate_lines[-1] is newest.
        """
        n = len(self.rate_lines)
        if n == 0:
            return np.array([])

        # age = 0 for newest, n-1 for oldest
        age = np.arange(n - 1, -1, -1, dtype=float)

        if self.half_life_lines <= 0:
            w = np.zeros(n)
            w[-1] = 1.0
            return w

        tau = self.half_life_lines / np.log(2.0)
        w = np.exp(-age / tau)
        w /= np.sum(w)
        return w

    def add_line(
        self,
        h_line,
        scan_speed_hat,
        dx_hat=None,
        valid=True,
        metadata=None,
    ):
        """
        Add one measured scan line.

        Parameters
        ----------
        h_line:
            Measured height line.

        scan_speed_hat:
            Scan speed used for this line.

        dx_hat:
            Pixel spacing. If None, uses self.dx_hat.

        valid:
            If False, line is ignored. Use this to skip invalid traces.

        metadata:
            Optional dictionary, e.g. drive, setpoint, i_gain, material.
        """
        if not valid:
            return

        h = np.asarray(h_line, dtype=float).ravel()
        dx = self.dx_hat if dx_hat is None else float(dx_hat)
        v = float(scan_speed_hat)

        if h.size < 4 or not np.isfinite(v) or v <= 0 or dx <= 0:
            return

        mask = np.isfinite(h)
        if np.mean(mask) < 0.9:
            return

        # Simple interpolation over rare non-finite values
        if not np.all(mask):
            x = np.arange(h.size)
            h = np.interp(x, x[mask], h[mask])

        dh = np.diff(h)
        rate = v * dh / dx

        rate = rate[np.isfinite(rate)]
        if rate.size < 3:
            return

        self.rate_lines.append(rate.astype(float))
        self.meta.append({} if metadata is None else dict(metadata))

        # Keep only most recent lines
        if len(self.rate_lines) > self.memory_size:
            self.rate_lines = self.rate_lines[-self.memory_size:]
            self.meta = self.meta[-self.memory_size:]

    def generate_line(
        self,
        n_pixels=256,
        scan_speed_hat=1.0,
        dx_hat=None,
        mode="block",
        block_size=16,
        preserve_zero_mean_rate=True,
        max_abs_rate=None,
        max_abs_dh=None,
        target_height_mad=None,
        start_height=0.0,
    ):
        """
        Generate a synthetic scan line from the weighted rate memory.

        Parameters
        ----------
        n_pixels:
            Number of height pixels to generate.

        scan_speed_hat:
            New scan speed for the synthetic line.

        mode:
            "line", "mixture", or "block".

        block_size:
            Block length for mode="block".

        preserve_zero_mean_rate:
            If True, subtract mean rate before integration to avoid line drift.

        max_abs_rate:
            Optional cap on |dh/dt|.

        max_abs_dh:
            Optional cap on local height jump |dh|.

        target_height_mad:
            Optional final roughness normalization.

        Returns
        -------
        dict with:
            h_hat
            rate
            dh
            weights
            selected_line_indices
        """
        if len(self.rate_lines) == 0:
            raise RuntimeError("RoughnessRateMemory is empty.")

        n_pixels = int(n_pixels)
        n_steps = n_pixels - 1
        if n_steps < 2:
            raise ValueError("n_pixels must be at least 3.")

        dx = self.dx_hat if dx_hat is None else float(dx_hat)
        v_new = float(scan_speed_hat)

        if v_new <= 0 or dx <= 0:
            raise ValueError("scan_speed_hat and dx_hat must be positive.")

        weights = self._decay_weights()
        n_lines = len(self.rate_lines)

        selected = []

        if mode == "line":
            line_idx = self.rng.choice(n_lines, p=weights)
            selected.append(int(line_idx))
            pool = self.rate_lines[line_idx]
            rate = self.rng.choice(pool, size=n_steps, replace=True)

        elif mode == "mixture":
            rate = np.empty(n_steps, dtype=float)
            for i in range(n_steps):
                line_idx = self.rng.choice(n_lines, p=weights)
                selected.append(int(line_idx))
                pool = self.rate_lines[line_idx]
                rate[i] = pool[self.rng.integers(0, len(pool))]

        elif mode == "block":
            rate_list = []

            while len(rate_list) < n_steps:
                line_idx = self.rng.choice(n_lines, p=weights)
                selected.append(int(line_idx))

                pool = self.rate_lines[line_idx]
                if len(pool) <= 1:
                    continue

                b = min(int(block_size), len(pool), n_steps - len(rate_list))
                if b <= 0:
                    break

                if len(pool) == b:
                    start = 0
                else:
                    start = self.rng.integers(0, len(pool) - b)

                rate_list.extend(pool[start:start + b])

            rate = np.asarray(rate_list[:n_steps], dtype=float)

        else:
            raise ValueError("mode must be 'line', 'mixture', or 'block'.")

        # Remove net slope/drift if desired
        if preserve_zero_mean_rate:
            rate = rate - np.nanmean(rate)

        # Optional rate cap
        if max_abs_rate is not None:
            max_abs_rate = float(max_abs_rate)
            if max_abs_rate > 0:
                rate = np.clip(rate, -max_abs_rate, max_abs_rate)

        # Convert rate back to local height step
        dh = rate * dx / v_new

        # Optional local height-jump cap
        if max_abs_dh is not None:
            max_abs_dh = float(max_abs_dh)
            if max_abs_dh > 0:
                dh = np.clip(dh, -max_abs_dh, max_abs_dh)
                rate = dh * v_new / dx

        h = np.empty(n_pixels, dtype=float)
        h[0] = float(start_height)
        h[1:] = h[0] + np.cumsum(dh)

        # Remove linear drift after integration
        x = np.linspace(0.0, 1.0, n_pixels)
        drift = h[0] + (h[-1] - h[0]) * x
        h = h - drift + h[0]

        if self.center_height:
            h = h - np.nanmedian(h)

        # Optional final roughness normalization
        if target_height_mad is not None:
            cur_mad = robust_mad(h)
            if cur_mad > 0:
                h = h * (float(target_height_mad) / cur_mad)

                # Recompute dh and rate after rescaling
                dh = np.diff(h)
                rate = v_new * dh / dx

                if max_abs_dh is not None:
                    dh = np.clip(dh, -max_abs_dh, max_abs_dh)
                    h[0] = 0.0
                    h[1:] = np.cumsum(dh)
                    h = h - np.nanmedian(h)
                    rate = v_new * np.diff(h) / dx

        return {
            "h_hat": h,
            "rate": rate,
            "dh": np.diff(h),
            "weights": weights,
            "selected_line_indices": np.asarray(selected, dtype=int),
            "mode": mode,
            "scan_speed_hat": v_new,
            "dx_hat": dx,
        }


def robust_mad(x, eps=1e-12):
    x = np.asarray(x, dtype=float).ravel()
    x = x[np.isfinite(x)]
    if x.size == 0:
        return eps
    med = np.nanmedian(x)
    return 1.4826 * np.nanmedian(np.abs(x - med)) + eps

In [ ]:
rough_mem = RoughnessRateMemory(
    memory_size=20,
    half_life_lines=5,
    dx_hat=1.0,
    random_state=0,
)

In [ ]:
# Example: add trace height line
rough_mem.add_line(
    h_line=h_exp_trace,
    scan_speed_hat=current_scan_speed,
    dx_hat=1.0,
    valid=True,
    metadata={
        "drive_nm": drive,
        "setpoint": setpoint,
        "i_gain": i_gain,
        "direction": "trace",
    },
)

# Optionally also add retrace
rough_mem.add_line(
    h_line=h_exp_retrace,
    scan_speed_hat=current_scan_speed,
    dx_hat=1.0,
    valid=True,
    metadata={
        "drive_nm": drive,
        "setpoint": setpoint,
        "i_gain": i_gain,
        "direction": "retrace",
    },
)

In [ ]:
pack = rough_mem.generate_line(
    n_pixels=256,
    scan_speed_hat=target_scan_speed,
    dx_hat=1.0,
    mode="block",
    block_size=16,
    max_abs_dh=None,
    max_abs_rate=None,
)

h_line_dt = pack["h_hat"]

In [ ]:
out = scanner.simulate_trace_retrace_substeps_fast(
    h_line_dt,
    drive_nm=drive,
    setpoint=setpoint,
    P=P_sim,
    I=I_sim,
    scan_speed_hat=target_scan_speed,
    dx_hat=1.0,
    n_substeps=50,
    fd_table=fd_table,
    h_smooth_sigma_px=0.0,
    d_init_hat=0,
    A_meas_init_hat=1,
)

In [ ]:
# predicted_metrics = []

# for q in range(8):
#     pack = rough_mem.generate_line(
#         n_pixels=256,
#         scan_speed_hat=target_scan_speed,
#         mode="block",
#         block_size=16,
#     )

#     out = run_dt(pack["h_hat"])
#     metrics = compute_metrics(out)
#     predicted_metrics.append(metrics)

In [ ]:
from codes.dt_fd_dynamic_residual_layers import (
    FDTableCache,
    ResidualDatasetConfig,
    StaticFDResidualConfig,
    DynamicResidualConfig,
    build_residual_dataset,
    fit_static_fd_residual_layer,
    add_static_predictions_and_dynamic_targets,
    fit_dynamic_residual_layer,
    predict_corrected_channels_for_out,
    residual_summary,
)

In [ ]:
from codes.dt_gp_static_calibration_v3 import predict_PI_for_full_grid

P_grid, I_grid, P_std_grid, I_std_grid, pred_rows = predict_PI_for_full_grid(
    gp_fit["gp_pack"],
    traces_exp,
    scan_speeds_hat=scan_rate_exp,
    drives_nm=drives_exp,
    setpoints=setpoints_exp,
    i_gains_exp=i_gain_exp,
)

In [ ]:
fd_cache = FDTableCache(
    scanner=scanner,
    n_d=scanner_cfg.fd_n_d,
)

resid_cfg = ResidualDatasetConfig(
    pairing="quantile_by_A",   # use "pixel" only if DT line and experiment are pixel-aligned
    max_points_per_direction=128,
    n_synthetic_lines_per_condition=1,
    random_state=0,

    phase_period_deg=360.0,    # change to 180 if your phase is 180-periodic

    stable_only=True,
    min_A_frac_of_A95=0.03,

    include_retrace=True,
    trim_edges=5,
)

df_base = build_residual_dataset(
    scanner,
    traces_exp_height=traces_exp,
    traces_exp_amp=traces_amp_exp,
    traces_exp_phase=traces_phase_exp,
    synthetic_lines=synthetic_lines,

    scan_speeds_hat=scan_rate_exp,
    drives_nm=drives_exp,
    setpoints=setpoints_exp,
    i_gains_exp=i_gain_exp,

    scanner_cfg=scanner_cfg,

    # From static GP calibration:
    P_grid=P_grid,
    I_grid=I_grid,

    fd_cache=fd_cache,
    cfg=resid_cfg,
)

print(df_base.shape)
df_base.head()

In [ ]:
static_cfg = StaticFDResidualConfig(
    feature_cols=("drive_nm", "d_dt"),
    target_cols=("delta_A_raw", "delta_phi_raw"),

    model_type="histgb",
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    min_samples_leaf=20,

    max_train_points=50000,
    random_state=0,
)

static_layer, static_info = fit_static_fd_residual_layer(
    df_base,
    cfg=static_cfg,
)

print(static_info)

In [ ]:
df_dyn = add_static_predictions_and_dynamic_targets(
    df_base,
    static_layer,
    fd_cache,
    phase_period_deg=360.0,
    use_target_deff_for_dynamic_targets=True,
)

print(df_dyn.shape)
df_dyn.head()

In [ ]:
dyn_cfg = DynamicResidualConfig(
    model_type="histgb",

    max_iter=400,
    learning_rate=0.04,
    max_leaf_nodes=31,
    min_samples_leaf=30,

    max_train_points=100000,
    random_state=0,
)

dynamic_layer, dynamic_info = fit_dynamic_residual_layer(
    df_dyn,
    cfg=dyn_cfg,
)

print(dynamic_info)

In [ ]:
si, di, spi, gi = 2, 8, 5, 4

drive = float(drive_exp[di])
fd_table = fd_cache.get(drive, drive_index=di)

P_sim = float(P_grid[si, di, spi, gi])
I_sim = float(I_grid[si, di, spi, gi])

out = scanner.simulate_trace_retrace_substeps_fast(
    h_line_dt,
    drive_nm=drive,
    setpoint=float(setpoint_exp[spi]),
    P=P_sim,
    I=I_sim,
    scan_speed_hat=float(scan_rate[si]),
    dx_hat=1.0,
    n_substeps=50,
    fd_table=fd_table,
    h_smooth_sigma_px=0.0,
    d_init_hat=0,
    A_meas_init_hat=1,
)

corrected = predict_corrected_channels_for_out(
    out,
    controls={
        "scan_speed_hat": float(scan_rate[si]),
        "drive_nm": drive,
        "setpoint": float(setpoint_exp[spi]),
        "I_exp": float(igain_exp[gi]),
        "P_sim": P_sim,
        "I_sim": I_sim,
    },
    fd_table=fd_table,
    static_layer=static_layer,
    dynamic_layer=dynamic_layer,
    static_at_deff=True,
)

In [ ]:
A_trace_pred = corrected["trace"]["A_pred"]
phi_trace_pred = corrected["trace"]["phi_pred"]

A_retrace_pred = corrected["retrace"]["A_pred"]
phi_retrace_pred = corrected["retrace"]["phi_pred"]

d_eff_trace = corrected["trace"]["d_eff"]

## Build a scanner from FD library

In [ ]:
from codes.Scanner_fixed import *
from codes.Scanner_numba_substeps import *

In [ ]:
# Load all the FD curves

data = np.load('output/cali_fd.npz')
# data = np.load('output/Tap300_AlScN.npz')

height = data['height']
phase = data['phase']
amp = data['amp']
drive = data['drive']

# plt.plot(amp[0])

drive_exp = np.zeros(len(drive))

for i in range(len(drive)):
    drive_exp[i] = np.mean(amp[i, -10:])

data = np.load('output/cleaned_traces.npz')
print(data.keys())

traces = data['traces']
x_measured = data['x_measured']
y_measured = data['y_measured']
topo = data['topo']


In [ ]:
drives_si = drive_exp
measured_fd = {}


# Assume these are lists/arrays where index i corresponds to drives[i]
# z_arrays[i], A_arrays[i], phi_arrays[i]
# each could be a 1D FD curve, a scan line, or an image
measured_fd = {
    float(drive): {
        "d": height[i],
        "A": amp[i],
        "phi": phase[i],
    }
    for i, drive in enumerate(drives_si)
}


measured_fd_ext, zero_amp_diag = extend_measured_fd_dict_linear_to_zero_amplitude(
    measured_fd,
    n_fit=5,
    n_bridge=50,
    d_zero=None,              # infer A=0 location from near-contact slope
    phi_mode="linear",
    F_mode="nearest",
    max_back_extrap_factor=5.0,
)

fd_surface = FDDriveSurface.from_measured_fd(
    measured_fd_ext,
    x_mode="d_over_A0",
    common_range="intersection",
    n_x=1000,
    far_field_frac=0.10,
    far_field_side="largest_d",
    drive_key_is_A0=False,
)

conv_L = 1.0  # or scanner.conversion["A"] if you are using normalized units


fd_model = make_normalized_fd_lookup(
    fd_surface,
    conv_L=1.0,
    conv_A=1.0,
    conv_F=1.0,
)



params = {
    "k": 25,
    "A": float(np.nanmedian(amp)),   # fallback only
    "d0": float(np.nanmax(height) * 0.8),
    "R": 10,
    "H": 1e-19,
    "E_star": 1e9,
    "Q": 250,
}

conversions = {k: 1.0 for k in params}

scanner = ScannerFD(
    params=params,
    conversions=conversions,
    fd_model=fd_model,
)

print("A, phi, F at drive 20:", scanner.measure_from_fd(50.0, drive_nm=20.0))
print("A, phi, F at drive 80:", scanner.measure_from_fd(50.0, drive_nm=80.0))
print("A0 at drive 20:", scanner.get_A0_hat(20.0))
print("A0 at drive 80:", scanner.get_A0_hat(80.0))
print("FD range at drive 20:", scanner.distance_range(20.0))

In [ ]:
drive_exp = x_measured[:,0]
setpoint_exp = x_measured[:,1] / x_measured[:,0]
igain_exp = x_measured[:,2]

traces = traces

traces_exp = np.zeros([60, 2, 256])

for i in range(60):
    for j in range(2):
        traces_exp[i,j] = (traces[i,-1,j]-np.min(traces[i,-1,j]))*1e9
    # traces_exp[i,2:] = traces[i,-1,2:]
idx = 10
h_truth = (traces[idx, -1, 0]-np.min(traces[idx, -1, 0]))*1e9
plt.plot(h_truth)

In [ ]:
fd20 = scanner.prepare_fd_table_for_drive(drive_nm=20, n_d=4096)

out = scanner.simulate_trace_retrace_substeps_fast(
    h_truth,
    drive_nm=20,
    setpoint=0.6,
    P=0.28*50,
    I=0.48*50,
    scan_speed_hat=1.0,
    dx_hat=1.0,
    n_substeps=50,
    fd_table=fd20,
    h_smooth_sigma_px=0.0,
)

fig, ax = plt.subplots(1,3,figsize=[10,3.])
ax[0].plot(out['trace']['d_hat'])
ax[0].plot(out['retrace']['d_hat'])
ax[2].plot(out['trace']['phi_deg'])
ax[2].plot(out['retrace']['phi_deg'])
ax[1].plot(out['trace']['A_hat'])
ax[1].plot(out['retrace']['A_hat'])
ax[0].set_ylabel('Height (nm)')
ax[1].set_ylabel('Amplitude (nm)')
ax[2].set_ylabel('Phase (degree)')
ax[0].set_title('Drive=20 nm, setpoint=0.6')
ax[1].set_title('P=0.28, I=0.48')

plt.tight_layout()